# KALYE AI Inference Server (Google Colab T4)

This notebook runs the AI pipeline (YOLOv8 + SegFormer + BLIP-2) on Colab's free T4 GPU
and exposes it as a FastAPI server via ngrok tunnel.

**Your local KALYE backend connects to the ngrok URL to get real AI detections.**

## Setup
1. Open in Google Colab
2. Runtime → Change runtime type → **T4 GPU**
3. Run all cells in order
4. Copy the **ngrok URL** printed at the end
5. Paste it into your local backend `.env` as `COLAB_AI_URL=https://xxxx.ngrok-free.app`
6. Restart your backend — uploads now use real AI!

## Cell 1: Install Dependencies

In [ ]:
!pip install -q ultralytics transformers accelerate fastapi uvicorn pyngrok Pillow python-multipart

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## Cell 2: Set your ngrok auth token

Get a **free** token at https://dashboard.ngrok.com/get-started/your-authtoken

Then paste it below.

In [ ]:
!pip install -q pyngrok

NGROK_AUTH_TOKEN = ""  # <-- PASTE YOUR NGROK TOKEN HERE

if not NGROK_AUTH_TOKEN:
    print("No ngrok token set! Get one free at https://dashboard.ngrok.com/get-started/your-authtoken")
    print("   Then set NGROK_AUTH_TOKEN above and re-run this cell.")
else:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("ngrok authenticated!")

## Cell 3: Load AI Models onto GPU

This downloads and loads:
- **YOLOv8n** — fast object detection (potholes, obstructions, etc.)
- **SegFormer-B3** — semantic segmentation (sidewalk coverage)
- **BLIP-2 OPT-2.7B** — image captioning

First run takes ~2-3 minutes to download. Subsequent runs use cached models.

In [ ]:
!pip install -q ultralytics transformers accelerate

import time
import torch
import numpy as np
from PIL import Image
from pathlib import Path

# Clear any leftover GPU memory from failed runs
torch.cuda.empty_cache()

# ── YOLOv8 ──────────────────────────────────────────────────────────
print("Loading YOLOv8...")
start = time.time()
from ultralytics import YOLO
yolo_model = YOLO("yolov8n.pt")
print(f"  YOLOv8n loaded in {time.time()-start:.1f}s")

COCO_TO_KALYE = {
    "car": "sidewalk_obstruction",
    "truck": "sidewalk_obstruction",
    "motorcycle": "sidewalk_obstruction",
    "bicycle": "sidewalk_obstruction",
    "stop sign": "missing_sign",
    "traffic light": "missing_sign",
    "fire hydrant": "sidewalk_obstruction",
    "parking meter": "sidewalk_obstruction",
    "bench": "sidewalk_obstruction",
    "person": "pedestrian",
    "dog": "sidewalk_obstruction",
    "potted plant": "sidewalk_obstruction",
    "umbrella": "street_vendor_obstruction",
    "suitcase": "sidewalk_obstruction",
    "backpack": "sidewalk_obstruction",
}

# ── SegFormer ───────────────────────────────────────────────────────
print("Loading SegFormer...")
start = time.time()
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor
seg_processor = SegformerImageProcessor.from_pretrained("nvidia/segformer-b3-finetuned-ade-512-512")
seg_model = SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b3-finetuned-ade-512-512")
seg_model = seg_model.to("cuda").eval()
seg_id2label = seg_model.config.id2label
print(f"  SegFormer loaded in {time.time()-start:.1f}s ({len(seg_id2label)} classes)")

# ── BLIP (base model — fits on T4 alongside other models) ──────────
print("Loading BLIP (base)...")
start = time.time()
from transformers import BlipForConditionalGeneration, BlipProcessor
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base", torch_dtype=torch.float16
).to("cuda")
print(f"  BLIP loaded in {time.time()-start:.1f}s")

# ── GPU memory check ────────────────────────────────────────────────
allocated = torch.cuda.memory_allocated() / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"\nGPU memory: {allocated:.1f} / {total:.1f} GB used")
print("All models loaded! Ready to serve.")

## Cell 4: Define the AI Pipeline Functions

In [ ]:
import torch.nn.functional as F

SIDEWALK_CLASS_ID = 11  # ADE20K: "sidewalk; pavement"
ROAD_CLASS_ID = 6       # ADE20K: "road; route"

def run_detection(image_path: str, confidence_threshold: float = 0.25) -> list:
    """Run YOLOv8 on an image, return KALYE-format detections."""
    results = yolo_model(image_path, verbose=False, conf=confidence_threshold)
    detections = []

    for r in results:
        if r.boxes is None:
            continue
        for i in range(len(r.boxes)):
            cls_id = int(r.boxes.cls[i].item())
            conf = float(r.boxes.conf[i].item())
            coco_name = yolo_model.names.get(cls_id, "")
            kalye_type = COCO_TO_KALYE.get(coco_name)

            if kalye_type is None or kalye_type == "pedestrian":
                continue

            xyxy = r.boxes.xyxy[i].tolist()
            detections.append({
                "detection_type": kalye_type,
                "confidence": round(conf, 3),
                "bounding_box": {
                    "x": round(xyxy[0], 1),
                    "y": round(xyxy[1], 1),
                    "w": round(xyxy[2] - xyxy[0], 1),
                    "h": round(xyxy[3] - xyxy[1], 1),
                },
                "coco_class": coco_name,
            })

    return detections


def run_segmentation(image_path: str) -> dict:
    """Run SegFormer, return sidewalk coverage and class breakdown."""
    image = Image.open(image_path).convert("RGB")
    inputs = seg_processor(images=image, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = seg_model(**inputs)

    logits = outputs.logits
    upsampled = F.interpolate(
        logits, size=(image.height, image.width), mode="bilinear", align_corners=False
    )
    mask = upsampled.argmax(dim=1).squeeze(0).cpu().numpy()

    total_px = mask.size
    unique, counts = np.unique(mask, return_counts=True)

    class_breakdown = {}
    for cls_id, count in zip(unique, counts):
        label = seg_id2label.get(cls_id, seg_id2label.get(str(cls_id), f"class_{cls_id}"))
        pct = round(float(count) / total_px * 100, 2)
        if pct >= 0.5:
            class_breakdown[label] = pct

    sidewalk_px = sum(c for cid, c in zip(unique, counts) if int(cid) == SIDEWALK_CLASS_ID)
    road_px = sum(c for cid, c in zip(unique, counts) if int(cid) == ROAD_CLASS_ID)

    return {
        "sidewalk_coverage_pct": round(float(sidewalk_px) / total_px * 100, 2),
        "road_coverage_pct": round(float(road_px) / total_px * 100, 2),
        "class_breakdown": class_breakdown,
    }


def run_captioning(image_path: str) -> str:
    """Run BLIP-base to generate a walkability-focused caption."""
    image = Image.open(image_path).convert("RGB")

    prompt = "a photograph of a street showing"
    inputs = blip_processor(images=image, text=prompt, return_tensors="pt").to("cuda", torch.float16)

    with torch.no_grad():
        generated_ids = blip_model.generate(**inputs, max_new_tokens=50)

    caption = blip_processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    if caption and caption[0].islower():
        caption = caption[0].upper() + caption[1:]
    if caption and not caption.endswith("."):
        caption += "."

    return caption


print("Pipeline functions defined.")

## Cell 5: Quick Test (Optional)

Upload a test image to verify the pipeline works before starting the server.

In [ ]:
# Upload a test image
from google.colab import files
print("Upload a street image to test the pipeline...")
uploaded = files.upload()

if uploaded:
    test_path = list(uploaded.keys())[0]
    print(f"\n--- Testing with: {test_path} ---")

    print("\n1. YOLOv8 Detection:")
    dets = run_detection(test_path)
    for d in dets:
        print(f"   {d['detection_type']} ({d['coco_class']}) — {d['confidence']:.0%}")
    if not dets:
        print("   No walkability issues detected (this is normal for clean streets)")

    print("\n2. SegFormer Segmentation:")
    seg = run_segmentation(test_path)
    print(f"   Sidewalk coverage: {seg['sidewalk_coverage_pct']}%")
    print(f"   Road coverage: {seg['road_coverage_pct']}%")
    for label, pct in sorted(seg['class_breakdown'].items(), key=lambda x: -x[1])[:5]:
        print(f"   {label}: {pct}%")

    print("\n3. BLIP-2 Caption:")
    caption = run_captioning(test_path)
    print(f"   {caption}")

    print("\nAll models working! You can now run Cell 6 to start the server.")

## Cell 6: Start the FastAPI Server + ngrok Tunnel

This is the main cell. It starts a FastAPI server that accepts image uploads
and returns real AI analysis results.

**Copy the ngrok URL** and paste it into your local `.env` file.

In [ ]:
import io
import os
import uuid
import tempfile
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
from pyngrok import ngrok

# ── FastAPI App ──────────────────────────────────────────────────────
app = FastAPI(title="KALYE AI Server", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/health")
async def health():
    gpu_mem = torch.cuda.memory_allocated() / 1024**3 if torch.cuda.is_available() else 0
    return {
        "status": "ok",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
        "gpu_memory_gb": round(gpu_mem, 2),
        "models_loaded": ["yolov8n", "segformer-b3", "blip2-opt-2.7b"],
    }


@app.post("/analyze")
async def analyze_image(file: UploadFile = File(...)):
    """Full AI pipeline: detection + segmentation + captioning.

    Returns all results in a single response so the local backend
    only needs one round-trip per image.
    """
    if file.content_type not in ("image/jpeg", "image/png", "image/webp"):
        raise HTTPException(400, f"Unsupported file type: {file.content_type}")

    # Save to temp file
    content = await file.read()
    suffix = ".jpg" if "jpeg" in file.content_type else ".png"
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
    tmp.write(content)
    tmp.close()
    tmp_path = tmp.name

    try:
        start = time.time()

        # Run all three models
        detections = run_detection(tmp_path)
        segmentation = run_segmentation(tmp_path)
        caption = run_captioning(tmp_path)

        elapsed = time.time() - start

        return {
            "status": "ok",
            "inference_time_seconds": round(elapsed, 2),
            "detections": detections,
            "segmentation": segmentation,
            "caption": caption,
        }

    finally:
        os.unlink(tmp_path)


# ── Start ngrok tunnel ───────────────────────────────────────────────
public_url = ngrok.connect(8000)
print("="*60)
print(f"KALYE AI Server is live!")
print(f"")
print(f"   Public URL: {public_url}")
print(f"")
print(f"Add this to your backend .env:")
print(f"   COLAB_AI_URL={public_url}")
print(f"")
print(f"Then restart your backend. Uploads will use real AI!")
print("="*60)

# ── Run server (blocks this cell) ────────────────────────────────────
uvicorn.run(app, host="0.0.0.0", port=8000)